# Fig.2b

In [ ]:
data = readRDS("cellbin.meta.rds")

In [ ]:
data=data[,c('area_m','Group','CellType','chip')]
data[which(data$CellType =="Fibroblast"),"CellType"]<-"Vascular cell"
data[which(data$CellType =="Blood"),"CellType"]<-"Vascular cell"
data[which(data$CellType =="Pericyte"),"CellType"]<-"Vascular cell"
data[which(data$CellType =="Ependy"),"CellType"]<-"Vascular cell"
data[which(data$CellType =="Endo"),"CellType"]<-"Vascular cell"
data[which(data$CellType =="IN_LAMP5"),"CellType"]<-"IN"
data[which(data$CellType =="IN_PVALB"),"CellType"]<-"IN"
data[which(data$CellType =="IN_SST"),"CellType"]<-"IN" 
data[which(data$CellType =="IN_VIP"),"CellType"]<-"IN" 

In [ ]:
# Calculate the count for each 'area_celltype'
count_mtx <- data %>%
  group_by(chip, area_celltype) %>%
  summarise(count = n()) %>%
  ungroup()

# Extract 'celltype' and 'area' from 'area_celltype'
count_mtx$celltype = gsub("-.*", "", count_mtx$area_celltype)
count_mtx$area = gsub(".*-", "", count_mtx$area_celltype)

# count_mtx=count_mtx[-which(count_mtx$area == 'NA'),]

# Calculate the total count for each 'area' within each 'Chips'
area_total_count <- count_mtx %>%
  group_by(chip, area) %>%
  summarise(total_count = sum(count)) %>%
  ungroup()

# Join the 'total_count' back to the 'count_mtx' based on 'Chips' and 'area'
count_mtx <- count_mtx %>%
  left_join(area_total_count, by = c("chip", "area"))

# Calculate the percentage for each 'area_celltype' within its 'area'
count_mtx <- count_mtx %>%
  mutate(percentage = count / total_count * 100)

# Now, 'count_mtx' contains the percentages of each 'area_celltype' within its 'area'
head(count_mtx)

In [ ]:
count_mtx1 = count_mtx
count_mtx1$Class  = '1'

## percentage
count_mtx1$Class[which(count_mtx1$chip %in% c('HS34',"HS22","HS33","HS29","HS27"))] <- 'Control'
count_mtx1$Class[which(count_mtx1$chip %in% c('HS36',"HS26","HS8","HS42C","HS42Y","HS43","HS50","HS45"))] <- 'Sclerosis'

count_mtx1$area_celltype_class = paste0(count_mtx1$Class,",",count_mtx1$area_celltype)

average_counts <- count_mtx1 %>%
  group_by(area_celltype_class) %>%
summarise(mean = mean(percentage))


average_counts$celltype = gsub("-.*", "",average_counts$area_celltype_class)
average_counts$area = gsub(".*-", "",average_counts$area_celltype_class)
average_counts$Class = gsub(",.*", "",average_counts$celltype )
average_counts$Celltype = gsub(".*,", "",average_counts$celltype)

In [ ]:
##比例
matrix_data <- matrix(0, nrow = length(unique(average_counts$area)), ncol = length(unique(average_counts$celltype)))

rownames(matrix_data) <- unique(average_counts$area)
colnames(matrix_data) <- unique(average_counts$celltype)

for (i in 1:nrow(average_counts)) {
  row_index <- which(rownames(matrix_data) == average_counts$area[i])
  col_index <- which(colnames(matrix_data) == average_counts$celltype[i])
  matrix_data[row_index, col_index] <- average_counts$mean[i]
}

matrix_data

In [ ]:
max(matrix_data)
##@##求取表达均值 气泡图大小
plot.data.ad=matrix_data[,12:ncol(matrix_data)]
plot.data.con=matrix_data[,1:11]
plot.data.mean<- data.frame(matrix(NA, nrow = nrow(plot.data.con), ncol = 11))
for (i in 1:11) {
  plot.data.mean[,i]=rowMeans(matrix_data[,c(i,11+i)])
  colnames(plot.data.mean)[i]=gsub("Control,","",colnames(plot.data.con)[i])
  
}
rownames(plot.data.mean)=rownames(matrix_data)


In [ ]:
plot.data.dis<- data.frame(matrix(NA, nrow = nrow(plot.data.con), ncol = 11))
#colnames(plot.data.dis)=names(table(result$area))
for (i in 1:nrow(plot.data.con)) {
  plot.data.dis[i,]=plot.data.ad[i,]-plot.data.con[i,]
  rownames(plot.data.dis)[i]=rownames(plot.data.con)[i]
}
colnames(plot.data.dis)=gsub("Sclerosis,","",colnames(plot.data.ad))

In [ ]:
order = c("EX_CA1","EX_CA2","EX_CA3","EX_DG","IN","Micro","Astro","Oligo","OPC","Vascular cell")
plot.data.dis <- plot.data.dis[,order ]
# matrix_data <- matrix_data[r_order ,]
plot.data.mean <- plot.data.mean[,order]

In [ ]:
dis.data=as.matrix(plot.data.dis)
expr.data=as.matrix(plot.data.mean)

dis.data.scale=as.data.frame(t(dis.data))
dis.data.scale=scale(dis.data.scale,center=F,scale=T)
dis.data.scale=as.data.frame(t(dis.data.scale))
dis.data.scale=as.matrix(dis.data.scale)

expr.data.scale=as.matrix((expr.data))

In [ ]:
mycolors <- c("#57121d", "#f1cdcc", "#eaebea", "#94c0d6", "#1f294e") 
col_fun = colorRamp2(c(-3,-1.5, 0, 1.5,3), rev(mycolors))

# 假设点大小的数值存储在 expr.data.scale 中
size_vector <- as.vector(expr.data.scale) # 确保你的点大小向量是正确的数据
max_size <- max(size_vector, na.rm = TRUE)
min_size <- min(size_vector, na.rm = TRUE)

# 自定义点大小图例
size_legend <- Legend(
  title = "Size",
  at = c(min_size, (max_size + min_size) / 2, max_size), # 调整图例范围
  labels = c("Small", "Medium", "Large"), # 图例标签
  legend_gp = gpar(fill = "black"), # 点的颜色
  size = unit(c(1, 2, 3), "mm") # 图例点大小
)

##############################################################################################################################
layer_fun = function(j, i, x, y, w, h, fill){
          grid.rect(x = x, y = y, width = w, height = h,
                    gp = gpar(col = NA, fill = NA))
          grid.circle(x=x,y=y,r= pindex(expr.data.scale, i, j)*0.03* unit(2, "mm"),
                      gp = gpar(fill = col_fun(pindex(dis.data.scale, i, j)), col = NA))}

# ##############################################################################################################################

# ha = rowAnnotation(typer1 = type,col = list(typer1=c("genome"="#2CA02CFF","MT.a"="#D62728FF","MT.b"="#9467BDFF"))
#                    #typet2 = anno_boxplot(expr.data*10, height = unit(1, "cm"), outline = FALSE,gp = gpar(fill = box.col))
#                   )
# # ##############################################################################################################################
column_ha =HeatmapAnnotation(df = data.frame(group=c("EX", "EX", "EX","EX", "IN", "Glial cell", "Glial cell", "Glial cell", "Glial cell", "Vasular"),row.names = colnames(dis.data)),
                            which = "column",
                            gap = unit(1,"mm"),
                            simple_anno_size = unit(0.45, "cm"),
                            col = list("group"=c("Glial cell"="#CB9BC7","EX"="#4DD7B1","IN"="#F66D81",'Vasular' = '#99bcac')))
# # ##############################################################################################################################
# right_annotation = HeatmapAnnotation(per = anno_barplot((expr.data), height = unit(3, "cm"), gp = gpar(fill = cluster_cols)),which = "row",show_legend = TRUE) #添加箱式图、修改其高度为1并为每一个样本的箱式图添加颜色
# top_annotation = HeatmapAnnotation(typet4 = anno_barplot(cell.number$Freq)) #添加箱式图、修改其高度为1并为每一个样本的箱式图添加颜色
# type=gene.info$group[match(rownames(dis.data),gene.info$gene)]

##############################################################################################################################
pdf("03.dotplot.pdf",width=7,height = 5)
ht <- ComplexHeatmap::Heatmap(dis.data.scale,
    # left_annotation = ha,
    top_annotation = column_ha,
    name = "hp",
    # right_annotation=right_annotation,
    heatmap_legend_param = list(title = "Diff."),
    col = col_fun,
    show_row_names = T,
   # bottom_annotation = column_ha,?
    row_title_gp = gpar(fontface = 'italic',fontsize = 15),
    rect_gp = gpar(type = "none"),
    layer_fun=layer_fun,
    # cell_fun = cell_fun,
    # row_split = factor(type),
    row_names_gp = gpar(fontsize = 10,fontface = 'italic'),
    column_split=factor(c("EX", "EX", "EX","EX", "IN", "Glial cell", "Glial cell", "Glial cell", "Glial cell", "Vasular"),levels = c("EX","IN","Glial cell",'Vasular')),
    cluster_columns = FALSE,
    cluster_rows = FALSE,
    border = "black")
draw(ht, annotation_legend_list = list(size_legend))
dev.off()
################

# Fig. 2d

In [ ]:
library(Seurat)
# library(SingleCellExperiment)
library(ggplot2)
library(cowplot)
library(RColorBrewer)
library(dplyr)
library(Matrix)
library(data.table)
library(reshape2)
library(tidyr)
summarySE <- function(data=NULL, measurevar, groupvars=NULL, na.rm=FALSE,
                      conf.interval=.95, .drop=TRUE) {
  library(plyr)
  
  # 计算长度
  length2 <- function (x, na.rm=FALSE) {
    if (na.rm) sum(!is.na(x))
    else       length(x)
  }
  
  # 以 groupvars 为组,计算每组的长度,均值,以及标准差
  # ddply 就是 dplyr 中的 group_by + summarise
  datac <- ddply(data, groupvars, .drop=.drop,
                 .fun = function(xx, col) {
                   c(N    = length2(xx[[col]], na.rm=na.rm),
                     mean = mean   (xx[[col]], na.rm=na.rm),
                     sd   = sd     (xx[[col]], na.rm=na.rm)
                   )
                 },
                 measurevar
  )
  
  # 重命名  
  datac <- plyr::rename(datac, c("mean" = measurevar))
  
  # 计算标准偏差
  datac$se <- datac$sd / sqrt(datac$N)  # Calculate standard error of the mean
  
  # Confidence interval multiplier for standard error
  # Calculate t-statistic for confidence interval: 
  # e.g., if conf.interval is .95, use .975 (above/below), and use df=N-1
  # 计算置信区间
  ciMult <- qt(conf.interval/2 + .5, datac$N-1)
  datac$ci <- datac$se * ciMult
  
  return(datac)
}

meta_d <- readRDS("cellbin.meta.rds")

meta_sub <- meta_d[which(meta_d$area_m %in% c("CA1","CA2","CA3","CA4","DG","FAS","SLRM")),]
meta_sub$CellType[which(grepl("IN",meta_sub$CellType))] = "IN"
# meta_sub$area_m <- "all"
meta_sub=meta_sub[,c("area_m","Group","CellType","chip_id")]
meta_sub$area_chip <- paste0(meta_sub$area_m,"_",meta_sub$chip_id)

df=as.data.frame(table(meta_sub$area_chip,meta_sub$CellType))
colnames(df)=c("area_chip","CellType","num")
df$area = gsub("_.*", "", df$area_chip)
df$chip = gsub(".*_", "", df$area_chip)

ex_map <- c("CA1" = "EX_CA1", "CA4" = "EX_CA3-4", "DG" = "EX_DG","CA2" = "EX_CA2","CA3" = "EX_CA3-4")
# 1) 取「本 region 对应的那一种 EX」
df_EX <- df %>%
  filter(CellType == ex_map[area]) %>%       
  select(area_chip, area, chip, EX = num)

# 2) 取 IN
df_IN <- df %>%
  filter(CellType == "IN") %>%
  select(area_chip, IN = num)

# 3) 一对一合并，求比值
df1 <- df_EX %>%
  inner_join(df_IN, by = "area_chip") %>%
  mutate(
    ratio = EX / IN,
    state = if_else(chip %in% c("HS22", "HS33", "HS34", "HS29"), "Normal", "HS"),
    area_state = paste0(area, "_", state)
  )

tgc <- summarySE(df1, measurevar="ratio", groupvars=c("area_state"))
tgc[is.na(tgc)]=0
tgc$area = gsub("_.*", "",tgc$area_state)
tgc$state = gsub(".*_", "",tgc$area_state)

p <- ggplot(tgc, aes(x = area, y = ratio, group = state, color = state)) +
  geom_line() +
  theme_minimal() +
  labs(title = "EX/IN Mean Ratio by Area and State", x = "Area", y = "Mean Ratio")+
  geom_point()+
  theme_bw()+
  theme(
    axis.title=element_text(size=15,face="plain",color="black"),
    axis.text = element_text(size=12,face="plain",color="black"),
    legend.title=element_text(size=14,face="plain",color="black"),
    legend.background  =element_blank())+
  # legend.position = c(0.88,0.88))
  scale_color_manual(values = c("Normal"="#8dc7c2", "HS"="#e94c5f"))+
  scale_y_continuous(breaks = seq(0, 15, 3))+
  geom_errorbar(aes(ymin=ratio-sd, ymax=ratio+sd), width=.1)+
  theme(panel.grid = element_blank(), panel.background = element_rect(color = 'black', fill = 'transparent')) 

In [ ]:
# calculate ratio of full celltype: EX/IN, EX/Astro

meta_sub <- meta[which(meta$area_m %in% c("CA1","CA2","CA3","CA4","DG","FAS","SLRM")),]
meta_sub$area_m <- "all"
meta_sub=meta_sub[,c("area_m","Group","CellType","chip_id")]
meta_sub <- meta_sub %>% drop_na()
meta_sub$area_chip <- paste0(meta_sub$area_m,"_",meta_sub$chip_id)

df=as.data.frame(table(meta_sub$area_chip,meta_sub$CellType))
colnames(df)=c("area_chip","celltype","num")
df$area = gsub("_.*", "", df$area_chip)
df$chip = gsub(".*_", "", df$area_chip)

df_EX=df[grepl('EX',which(df$celltype)),]
df_IN=df[grepl('IN',which(df$celltype)),]
# df_astro=df[which(df$celltype == "Astro"),]

df1=merge(df_EX,df_IN,by="area_chip")
df1$ratio=df1$num.x/df1$num.y
df1$state="HS"
df1$state[which(df1$chip.x %in% c("HS22","HS33","HS34","HS29"))] <- "Normal"
df1=df1[,c("ratio","state","area.x","chip.x")]
df1$area_state=paste0(df1$area,"_",df1$state)
df1$chip_area_state=paste0(df1$chip.x,"_",df1$area_state)
head(df1)

tgc <- summarySE(df1, measurevar="ratio", groupvars=c("area_state"))
tgc$area = gsub("_.*", "", tgc$area_state)
tgc$state = gsub(".*_", "", tgc$area_state)
tgc$state <- factor(tgc$state)

normal_ratio <- df1$ratio[df1$state == "Normal"]
sclerosis_ratio <- df1$ratio[df1$state == "HS"]

t_test_result <- t.test(normal_ratio, sclerosis_ratio)
t_test_result$p.value

options(repr.plot.width=5, repr.plot.height=5)   
comparisons <- list(c("Normal", "HS"))
p0 = ggplot(tgc,mapping =aes(x = factor(state,levels = c("Normal","HS")),y=ratio,fill = state))+
  geom_bar(stat="summary",fun = 'mean',width=0.8,color="black",fun.args = list(mult=1))+
  scale_fill_manual(values = c("#e94c5f","#8dc7c2"))+
  stat_summary(fun.data='mean_sdl',fun.args = list(mult=1),geom="errorbar",width=0.2,colour="black")+
  theme_minimal()+
  theme_bw()+
  theme(panel.grid = element_blank(), panel.background = element_rect(color = 'black', fill = 'transparent')) + #去掉背景的格子线
  theme(axis.text.x = element_text(size=14),axis.text.y = element_text(size=14),legend.title=element_text(size=14),legend.text=element_text(size=14),
        axis.title=element_text(size = 16))+ylab("EX:Astro ratio")+
  geom_errorbar(aes(ymin=ratio-se, ymax=ratio+se), width=.1)
# theme(panel.grid = element_blank(), panel.background = element_rect(color = 'black', fill = 'transparent'))
print(p0)
ggsave(plot=p0,"EX.IN.ratio.full.pdf",width=5,height=5)

# Fig.2l

In [ ]:
up <- fread("/data/work/Spatial_analysis/cellbin/07.Final/01.fig2/02.allcelltype.Sclerosis-up.GOterm.all.txt")
up$group="up"
down <- fread("/data/work/Spatial_analysis/cellbin/07.Final/01.fig2/02.allcelltype.Sclerosis-down.GOterm.all.txt")
down$group="down"
all=rbind(up,down)

GO.combined.top.BP <- all[which(all$Description %in% c('leukocyte migration',
                                                       'mitochondrial ATP synthesis coupled electron transport',
                                                       'regulation of synaptic plasticity',
                                                       'calcium ion transport',
                                                       'synaptic vesicle cycle',
                                                       'response to metal ion',
                                                       'aging',
                                                       'protein localization to cell periphery',
                                                       'cell junction assembly',
                                                       'neuron projection extension',
                                                       'neurotransmitter secretion',
                                                       'cellular oxidant detoxification',
                                                       'positive regulation of ERK1 and ERK2 cascade',
                                                       'gliogenesis')),]
up <- GO.combined.top.BP[which(GO.combined.top.BP$group=="up"),]
mat_up <- up[,c("qvalue","Description","cluster")]

mat_up=mat_up[order(mat_up$Description,decreasing=T),]
mat_up_wider <- mat_up %>% 
  pivot_wider(
    names_from = cluster,
    values_from = qvalue
  )
mat_up_wider=as.matrix(mat_up_wider)
rownames(mat_up_wider)=mat_up_wider[,1]
mat_up_wider=mat_up_wider[,-1]
mat_up_wider[is.na(mat_up_wider)]=1
mat_up_wider

# dg=matrix(1,ncol=1,nrow=18)
# colnames(dg)=c("Blood")
# mat_up_wider_up=cbind(mat_up_wider,dg)

# learning_or_memory <- c(1,1,1,1,1,1,1,1,1)
cell_junction_assembly <- c(1,1,1,1,1,1,1,1,1)
protein_localization_to_cell_periphery <- c(1,1,1,1,1,1,1,1,1)
synaptic_vesicle_cycle <- c(1,1,1,1,1,1,1,1,1)
neurotransmitter_secretion <- c(1,1,1,1,1,1,1,1,1)

mat_up_wider_up=rbind(mat_up_wider,cell_junction_assembly,protein_localization_to_cell_periphery,synaptic_vesicle_cycle,neurotransmitter_secretion)
mat_up_wider_up=mat_up_wider_up[,c("EX_CA1","EX_CA2-4","EX_DG","IN","Astro","Oligo","OPC","Micro","Vascular")]
mat_up_wider_up=mat_up_wider_up[rev(c('response to metal ion',
                                      'aging',
                                      'mitochondrial ATP synthesis coupled electron transport',
                                      'cellular oxidant detoxification',
                                      'gliogenesis',
                                      'leukocyte migration',
                                      'positive regulation of ERK1 and ERK2 cascade',
                                      'regulation of synaptic plasticity',
                                      'calcium ion transport',
                                      'neuron projection extension',
                                      'synaptic_vesicle_cycle',
                                      'neurotransmitter_secretion',
                                      'protein_localization_to_cell_periphery',
                                      'cell_junction_assembly')),]
mat_up_wider_up

mat_long <- reshape2::melt(mat_up_wider_up,var.id="GO_term")
mat_long$value = as.numeric(mat_long$value)
mat_long$value=-log10(mat_long$value)


options(repr.plot.width = 12, repr.plot.height = 8)
p1<-ggplot(mat_long,aes(x=Var2,y=Var1,fill=value)) #热图绘制
p2 <- p1+geom_raster()+scale_fill_gradientn(colors = c("white","#EFBDBD","#EF9E9E","#D35050","#DB4C5C"),breaks=c(0,3,6,9,12))+
  theme_bw()+
  theme(axis.text.x=element_text(angle=90,hjust = 1,vjust=0.5),
        axis.text.y=element_text(size=12),
        axis.text = element_text(color = 'black', size = 11))+
  # scale_fill_manual(values = c('#E2DFD0',"#F8E559","#864AF9"))+
  labs(title="up-regulate GO",xlab=NULL)
ggsave("03.cellbin.up-regulate-GO.pdf",p2,width=10)


# Fig.2k

In [ ]:
library(rrvgo)
library("clusterProfiler")
library("org.Hs.eg.db")
library("enrichplot")

setwd("/data/work/01.result/01.cellbin.deg")
de.all = read.table("01.01.Sclerosis_vs_Normal.allgenes.240820.txt",sep="\t",check.names = F,header = T)
# table(de.all$cluster,de.all$up.down)
# de.all=de.all[de.all$p_val_adj<0.05,]
# de.up=de.all[de.all$up.down=="up",]
GO.combined = c()
for(i in unique(de.all$cluster)){
  temp = de.all[which(de.all$cluster %in% i),]
  #for(i in unique(paste0(DE.combine$class,"_",DE.combine$up.down))){
  #temp = DE.combine[which(paste0(DE.combine$class,"_",DE.combine$up.down) %in% i),]
  #mygene <- AnnotationDbi::select(org.Hs.eg.db,columns=c("SYMBOL","ENTREZID"),keytype="SYMBOL",keystype="SYMBOL",keys=temp$gene)
  mygene <- tryCatch(
    { AnnotationDbi::select(org.Hs.eg.db,columns=c("SYMBOL","ENTREZID"),keys=temp$gene,keytype="SYMBOL") },
    warning = function(w) { message('Waring @ ',temp$gene) ; return(NA) },
    error = function(e) { message('Error @ ',temp$gene) ; return(NA) },
    finally = { message('next...') }
  )
  if (is.na(data.frame(mygene)[1,1])=="FALSE"){
    mygene<-mygene$ENTREZID
    ego <- enrichGO(gene = mygene,OrgDb = org.Hs.eg.db,ont = "ALL",pAdjustMethod = "BH",pvalueCutoff = 1,qvalueCutoff = 1,readable = TRUE)
    GO.list = ego@result
    GO.list$cluster = i
    GO.combined = rbind(GO.combined,GO.list)
  }else{print("None of the keys entered are valid keys for 'SYMBOL:")}
  print(i)
}
write.table(GO.combined,"01.allDEG.GOterm.all.txt",sep="\t",row.names=F,quote=F)
GO.combined.filter = GO.combined[GO.combined$p.adjust < 0.05,]

simMatrix <- calculateSimMatrix(GO.combined.filter$ID, orgdb="org.Hs.eg.db", ont="BP", method="Rel")
scores <- setNames(-log10(GO.combined.filter$qvalue), GO.combined.filter$ID)
reducedTerms <- reduceSimMatrix(simMatrix, scores, threshold=0.9, orgdb="org.Hs.eg.db")

pdf("03.rrvgo.pdf",width=7,height=7)
treemap::treemap(reducedTerms, index = c("parentTerm", "term"), 
                 vSize = 'score', type = "index", title = '', palette = acols)
dev.off()
